In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# How to Check that a Formula is a Tautology

In this notebook we develop a function <tt>tautology</tt> that takes a formula $f$ from propositional logic and checks whether $f$ is a tautology.  As we want to write formulas as strings, we first have to import the parser for propositional logic.

In [2]:
importNotebook "Propositional-Logic-Parser.ipynb"

imported 

Propositional-Logic-Parser.ipynb

As we represent propositional valuations as sets of variables, we need a function to compute all subsets of a given set.  

The function `allSubsets` takes one argument:
- `M` is a finite set whose elements have the generic type `'T`.

It returns a list containing all subsets of `M`, that is we have:
$$ \texttt{allSubsets}(M) = \bigl[A \mid A \in 2^M\bigr] $$

**Implementation:** If `M` is empty, the only subset is the empty set.  Otherwise, we remove an element `x` from `M` and compute the list `L` of all subsets of the remaining set recursively.  The subsets of `M` are the subsets in `L` together with the sets that result from adding `x` to the subsets in `L`.

In [3]:
let rec allSubsets (M: Set<'T>) : Set<'T> list =
    if M.IsEmpty then
        [ Set.empty ]
    else
        let x = Set.minElement M
        let L = allSubsets (M.Remove x)
        L @ [ for K in L -> K.Add x ]

In [4]:
allSubsets (set [ "p"; "q" ])

[set []; set ["q"]; set ["p"]; set ["p"; "q"]]

A set with $n$ elements has $2^n$ subsets.  Hence, the number of subsets grows very fast.  For a set with $20$ elements, there are already more than a million subsets.  The function `List.length` returns the length of a list.

In [5]:
#!time
List.length (allSubsets (set [1 .. 20]))

1048576

Wall time: 236.505ms

In [6]:
allSubsets (set [ "p"; "q"; "r" ])

[set []; set ["r"]; set ["q"]; set ["q"; "r"]; set ["p"]; set ["p"; "r"]; set ["p"; "q"]; set ["p"; "q"; "r"]]

To be able to compute all propositional valuations for a given formula $f$ we first need to determine the set of all variables that occur in $f$.  

The function `collectVars` takes one argument:
- `f` is a propositional formula.

It returns the set of all propositional variables occurring in `f`.  This function is defined recursively.  The operator `+` computes the union of two sets.

In [7]:
let rec collectVars (f: Formula) : Set<string> =
    match f with
    | Var p            -> set [ p ]
    | Verum | Falsum   -> Set.empty
    | Not g            -> collectVars g
    | And (g, h) | Or (g, h) | Imp (g, h) | Iff (g, h) | Xor (g, h) 
                       -> collectVars g + collectVars h

The function `evaluate` takes two arguments:
- `f` is a propositional formula,
- `I` is a propositional valuation, represented as the set of those variables that are true.

It returns the value of `f` given the valuation `I`.  We have discussed this function previously.  Here, the implication $g \rightarrow h$ is evaluated using the equivalence $(g \rightarrow h) \Leftrightarrow \neg g \vee h$.

In [8]:
let rec evaluate (f: Formula) (I: Set<string>) : bool =
    match f with
    | Var p      -> I.Contains p
    | Verum      -> true
    | Falsum     -> false
    | Not g      -> not (evaluate g I)
    | And (g, h) -> evaluate g I && evaluate h I
    | Or  (g, h) -> evaluate g I || evaluate h I
    | Imp (g, h) -> not (evaluate g I) || evaluate h I
    | Iff (g, h) -> evaluate g I =  evaluate h I
    | Xor (g, h) -> evaluate g I <> evaluate h I

Now we are ready to define the function `tautology`.  It takes one argument:
- `f` is a propositional formula.

The function checks whether $f$ is a tautology.  In Python, this function returns either `True` or a counter example.  In F#, every function has a unique result type.  Therefore, we use the type `Set<string> option`.  A value of this type has one of two forms:
- `None` indicates that `f` is a tautology,
- `Some I` indicates that `f` is not a tautology.  Then `I` is a set of variables such that $f$ evaluates to `false` if all variables in $I$ are `true`, while all variables not in $I$ are `false`.

**Implementation:** `List.tryFind` returns `Some I` for the first valuation `I` in the list of all valuations such that `f` evaluates to `false`.  If there is no such valuation, `None` is returned.

In [9]:
let tautology (f: Formula) : Set<string> option =
    let P = collectVars f
    List.tryFind (fun I -> not (evaluate f I)) (allSubsets P)

The function `test` takes one argument:
- `s` is a string that can be parsed as a propositional formula.

It checks whether this formula is a tautology and prints the result.  If the formula is not a tautology, a counter example is printed.

In [10]:
let test (s: string) : unit =
    let f = parse s
    match tautology f with
    | None ->
        printfn "The formula %s is a tautology." s
    | Some counterExample ->
        printfn "The formula %s is not a tautology." s
        printfn "Counter example: "
        for x in collectVars f do
            if counterExample.Contains x then
                printfn "%s ↦ true" x
            else
                printfn "%s ↦ false" x

Let us run a few tests.

The first example is DeMorgan's rule.

In [11]:
test "¬(p ∨ q) ↔ ¬p ∧ ¬q"

The formula 

¬(p ∨ q) ↔ ¬p ∧ ¬q

 is a tautology.

In [12]:
test "(p → q) → (¬p → q) → q"

The formula 

(p → q) → (¬p → q) → q

 is a tautology.

In [13]:
test "(p → q) → (¬p → ¬q)"

The formula 

(p → q) → (¬p → ¬q)

 is not a tautology.

Counter example: 

p

 ↦ false

q

 ↦ true

In [14]:
test "(p → q) → (¬q → ¬p)"

The formula 

(p → q) → (¬q → ¬p)

 is a tautology.

In [15]:
test "¬p ↔ (p → ⊥)"

The formula 

¬p ↔ (p → ⊥)

 is a tautology.